# L1 · Likelihood: from one observation to a dataset

This notebook follows the lecture slowly:

1. Fix one observed value and compare candidate parameters.
2. Repeat for a discrete and a continuous model.
3. Separate **independent** from **identically distributed**.
4. Use independence to factor a joint likelihood into per-example terms.
5. Verify that products become sums in log space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (7, 3.6),
    'axes.spines.top': False,
    'axes.spines.right': False,
})
ORANGE, TEAL, GREY = '#E98313', '#268582', '#7C898D'

## 1 · One discrete observation

A Bernoulli model says $P(Y=1\mid\theta)=\theta$. After observing the fixed value $y^*=1$, the likelihood is

$$L(\theta;y^*=1)=P(Y=1\mid\theta)=\theta.$$

Before running the cell: which candidate has higher likelihood, $\theta=0.3$ or $\theta=0.8$?

In [ ]:
theta_candidates = np.array([0.3, 0.8])
likelihoods = theta_candidates  # because the fixed observation is y*=1
for theta, likelihood in zip(theta_candidates, likelihoods):
    print(f'theta={theta:.1f}: L(theta)={likelihood:.3f}')

theta_grid = np.linspace(0, 1, 201)
plt.plot(theta_grid, theta_grid, color=ORANGE, lw=2)
plt.scatter(theta_candidates, likelihoods, color=[GREY, TEAL], zorder=3)
plt.xlabel('candidate coin bias theta')
plt.ylabel('likelihood L(theta)')
plt.title('One observed head: L(theta) = theta')
plt.show()

## 2 · One continuous observation

Let $Y\mid\mu\sim\mathcal N(\mu,1)$ and observe $y^*=1$. Now $L(\mu)$ is the **density at the fixed observation**, viewed as a function of the candidate mean $\mu$. It is not the probability of the exact point.

In [ ]:
def normal_pdf(y, mu, sigma=1.0):
    return np.exp(-0.5 * ((y - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

y_star = 1.0
mu_candidates = np.array([0.0, 1.0, 2.0])
for mu in mu_candidates:
    print(f'mu={mu:.1f}: L(mu)={normal_pdf(y_star, mu):.3f}')

mu_grid = np.linspace(-2, 4, 301)
plt.plot(mu_grid, normal_pdf(y_star, mu_grid), color=ORANGE, lw=2)
plt.scatter(mu_candidates, normal_pdf(y_star, mu_candidates), color=TEAL, zorder=3)
plt.axvline(y_star, color=GREY, ls='--', label='observed y*=1')
plt.xlabel('candidate mean mu')
plt.ylabel('likelihood L(mu)')
plt.title('One Gaussian observation: the best centre is the observation')
plt.legend()
plt.show()

## 3 · Independent and identical are different claims

- **Independent given $\theta$:** once the parameter is fixed, one outcome gives no extra information about another. This is what permits multiplication.
- **Identically distributed:** every observation uses the same sampling rule and shared parameter.

Two separate coins with biases $\theta_1$ and $\theta_2$ can be independent but not identically distributed. Their joint probability still factorizes: $P(H,H)=\theta_1\theta_2$.

In [ ]:
theta_1, theta_2 = 0.8, 0.3
print('Independent, not identical:')
print(f'P(H on coin 1, H on coin 2) = {theta_1} x {theta_2} = {theta_1 * theta_2:.2f}')

theta = 0.8
print('\nDependent copied outcome Y2=Y1:')
print(f'P(H,H) = P(Y1=H) = {theta:.2f}, not theta^2 = {theta**2:.2f}')

## 4 · Two tiny Bernoulli datasets

Compare a fair coin ($\theta=0.5$) and a head-biased coin ($\theta=0.8$). Predict the winner for the datasets `H,T` and `H,H` before running the cell.

In [ ]:
def bernoulli_likelihood(data, theta):
    data = np.asarray(data)
    terms = theta**data * (1 - theta)**(1 - data)
    return terms, terms.prod()

datasets = {'H,T': [1, 0], 'H,H': [1, 1]}
for name, data in datasets.items():
    print(f'\nObserved {name}')
    for theta in [0.5, 0.8]:
        terms, joint = bernoulli_likelihood(data, theta)
        print(f'  theta={theta:.1f}: terms={terms}, product={joint:.3f}')

## 5 · A likelihood curve from many flips

For i.i.d. flips with $n_H$ heads and $n_T$ tails,

$$L(\theta)=\theta^{n_H}(1-\theta)^{n_T},\qquad \log L(\theta)=n_H\log\theta+n_T\log(1-\theta).$$

In [ ]:
data = np.array([1, 1, 0, 0, 0, 1, 1, 0, 0, 0])  # 4 heads, 6 tails
theta_grid = np.linspace(0.001, 0.999, 500)
likelihood = np.array([bernoulli_likelihood(data, theta)[1] for theta in theta_grid])
log_likelihood = np.log(likelihood)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(theta_grid, likelihood, color=ORANGE)
axes[0].set(title='Likelihood', xlabel='theta', ylabel='L(theta)')
axes[1].plot(theta_grid, log_likelihood, color=TEAL)
axes[1].set(title='Log-likelihood', xlabel='theta', ylabel='log L(theta)')
for ax in axes:
    ax.axvline(data.mean(), color=GREY, ls='--', label=f'head fraction={data.mean():.1f}')
    ax.legend()
plt.tight_layout()
plt.show()
print('argmax likelihood    =', theta_grid[np.argmax(likelihood)].round(3))
print('argmax log-likelihood=', theta_grid[np.argmax(log_likelihood)].round(3))

## 6 · Two continuous measurements

Let $Y_i\mid\mu\sim\mathcal N(\mu,1)$ independently and observe $y=(1,2)$. We multiply **densities**.

In [ ]:
y = np.array([1.0, 2.0])
for mu in [1.5, 0.0]:
    terms = normal_pdf(y, mu)
    likelihood = terms.prod()
    log_likelihood = np.log(terms).sum()
    print(f'mu={mu:.1f}: terms={terms.round(3)}, product={likelihood:.3f}, sum(log terms)={log_likelihood:.2f}')

## 7 · Your turn

1. Change the coin data to `H,H,H,T`. Where should the likelihood peak? Verify it.
2. For Gaussian observations $y=(0,2,4)$ with known $\sigma=1$, predict the MLE of $\mu$, then plot $L(\mu)$.
3. Construct two independent but non-identical Bernoulli observations and write their joint likelihood.
4. Explain why identical distribution is **not** the step that permits multiplication.

## Takeaways

- Likelihood fixes the observed data and varies the model parameters.
- The likelihood of one continuous observation is a density value.
- Independence factorizes the joint likelihood; a shared sampling rule supplies the common parameterization.
- Log-likelihood adds the per-example log terms and has the same maximizer as likelihood.